In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,
                             classification_report,
                             confusion_matrix,
                             f1_score,
                             ConfusionMatrixDisplay)

print('Model training and evaluation tools imported successfully!')

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [ ]:
data_path = "../data/raw/data.csv"

df = pd.read_csv(data_path, sep=";")

print("Dataset loaded successfully!")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
missing_values = df.isnull().sum()

print("Total missing values:", missing_values.sum())

print("\nColumns containing missing values:")
print(missing_values[missing_values > 0])

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
constant_columns = [
    column
    for column in df.columns
    if df[column].nunique() == 1
]

print("Columns with only one unique value:", constant_columns)

In [ ]:
df.Target.value_counts()

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.dtypes

In [ ]:
df['Target'].value_counts(normalize=True).mul(100).round(2)

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(
    data=df,
    x='Target',
    order=df['Target'].value_counts().index
)

plt.title('Deistribution of Student Outcomes.')
plt.xlabel('Student Outcomes')
plt.ylabel('Number of Students')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
for column in df.columns:
    print(f"{column:45} {df[column].dtype}")

In [ ]:
df.describe().T

In [ ]:
# Define the target variable
target_column = "Target"

# Columns unavailable at our intended prediction point
excluded_columns = [
    "Curricular units 2nd sem (credited)",
    "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Curricular units 2nd sem (without evaluations)",
]



In [ ]:
# Create features and target
X = df.drop(columns=[target_column] + excluded_columns)
y = df[target_column]

In [ ]:
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget classes:")
print(y.unique())

print("\nRemaining feature columns:")
print(X.columns.tolist())

In [ ]:
print("Number of input features:", X.shape[1])
print("Number of student records:", X.shape[0])

print("\nTarget distribution:")
print(y.value_counts())

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

In [ ]:
categorical_columns = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International",
]

categorical_columns = [
    column for column in categorical_columns
    if column in X_train.columns
]

numerical_columns = [
    column for column in X_train.columns
    if column not in categorical_columns
]

In [ ]:
print("Categorical features:", len(categorical_columns))
print("Numerical features:", len(numerical_columns))

In [ ]:
print("\nCategorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_columns
        ),
    ]
)

In [ ]:
model = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        (
            'classifier',
            LogisticRegression(max_iter=2000, random_state=42)
        )
    ]
)

print('Model Pipeline created successfully!')

In [ ]:
model.fit(X_train,y_train)
print('Model training completed successfully!')

In [ ]:
y_pred = model.predict(X_test)


In [ ]:
print('Predictions generated successfully!')
print('\nFirst 10 actual outcomes: ')
print(y_test.head(10).tolist())

print('\nFirst 10 Predicted outcomes: ')
print(y_pred[:10].tolist())

In [ ]:
accuracy = accuracy_score(y_test,y_pred)
balanced_accuracy = balanced_accuracy_score(y_test,y_pred)
macro_f1 = f1_score(
    y_test,
    y_pred,
    average='macro'
)

print(f"Accuaracy:   {accuracy:.4f}")
print(f"Balanced Accuracy:    {balanced_accuracy:.4f}")
print(f"Macro F1-score:     {macro_f1:.4f}")

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        labels=['Dropout','Enrolled','Graduate'],
        zero_division=0
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=['Dropout','Enrolled','Graduate'],
    cmap="Blues",
    values_format='d'
)

plt.title('Logistic Regression - Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
rf_model = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        (
            'classifier',
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(X_train,y_train)
print('Random Forest training completed successfully!')

In [ ]:
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test,rf_pred)
rf_balanced_accuracy = balanced_accuracy_score(y_test,rf_pred)
rf_macro_f1 = f1_score(y_test,rf_pred,average='macro')

print(f"Accuaracy:   {rf_accuracy:.4f}")
print(f"Balanced Accuracy:    {rf_balanced_accuracy:.4f}")
print(f"Macro F1-score:     {rf_macro_f1:.4f}")

print(
    classification_report(
        y_test,
        rf_pred,
        labels=['Dropout','Enrolled','Graduate'],
        zero_division=0
    )
)

In [ ]:
results = pd.DataFrame({
    'Model':[
        'Logistic Regression',
        'Random Forest'
    ],
    'Accuracy':[
        accuracy,
        rf_accuracy
    ],
    'Balanced Accuracy':[
        balanced_accuracy,
        rf_balanced_accuracy
    ],
    'Macro F1-Score':[
        macro_f1,
        rf_macro_f1
    ]
})

results = results.set_index('Model')
results.round(4)

In [ ]:
results.plot(
    kind='bar',
    figsize=(10,6),
    ylim=(0,1),
    rot=0
)

plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xlabel('Model')
plt.legend(loc='lower right')
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    rf_pred,
    labels=['Dropout','Enrolled','Graduate'],
    display_labels=['Dropout','Enrolled','Graduate'],
    cmap="Blues",
    values_format='d'
)

plt.title('Random Forest - Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
print(
    classification_report(
        y_test,
        rf_pred,
        labels=['Dropout','Enrolled','Graduate'],
        target_names=['Dropout','Enrolled','Graduate'],
        zero_division=0
    )
)

In [ ]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nUnique target classes:")
print(y_train.unique())

print("\nTraining rows:", len(y_train))

In [ ]:
# check overfitting 

#Logistic Regression
lr_train_pred = model.predict(X_train)
lr_test_pred = model.predict(X_test)

#Random Forest
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)

In [ ]:
Overfitting_results = pd.DataFrame({
    "Model":['Logistic Regression', 'Random Forest'],
    'Train Accuracy': [
        accuracy_score(y_train,lr_train_pred),
        accuracy_score(y_train, rf_train_pred)
    ],
    'Test Accuracy': [
        accuracy_score(y_test,lr_test_pred),
        accuracy_score(y_test,rf_test_pred)
    ],
    'Train Macro F1': [
        f1_score(y_train, lr_train_pred,average='macro'),
        f1_score(y_train, rf_train_pred, average='macro')
    ],
    'Test Macro F1': [
        f1_score(y_test, lr_test_pred,average='macro'),
        f1_score(y_test, rf_test_pred, average='macro')
    ]
}).set_index('Model')

Overfitting_results.round(3)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)
print("Number of folds:", cv.get_n_splits())

scoring = {
    'accuracy':'accuracy',
    'balanced_accuracy':'balanced_accuracy',
    'macro_f1':make_scorer(
        f1_score,
        average='macro'
    )
}


In [ ]:
cv_results = cross_validate(
    estimator = model,
    X = X_train,
    y = y_train,
    cv = cv,
    scoring=scoring,
    # groups=groups_train,
    n_jobs=-1,
    return_train_score=True
)

cv_summary = pd.DataFrame({
    'Metric':[
        'Accuracy',
        'Balanced Accuracy',
        'Macro F1'
    ],
    "Mean CV Score": [
        cv_results['test_accuracy'].mean(),
        cv_results['test_balanced_accuracy'].mean(),
        cv_results['test_macro_f1'].mean()
    ],
    "Standard Deviation": [
        cv_results['test_accuracy'].std(),
        cv_results['test_balanced_accuracy'].std(),
        cv_results['test_macro_f1'].std()
    ]
})

cv_summary.round(3)

In [ ]:

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42,
                class_weight="balanced",
                n_jobs=1
            )
        )
    ]
)

param_distributions = {
    "classifier__n_estimators": [150, 250, 350],
    "classifier__max_depth": [None, 10, 20, 30],
    "classifier__min_samples_split": randint(2, 15),
    "classifier__min_samples_leaf": randint(1, 8),
    "classifier__max_features": ["sqrt", "log2"]
}

In [ ]:
rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="f1_macro",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

rf_search.fit(X_train, y_train)

print("Best cross-validation Macro F1:", round(rf_search.best_score_, 4))
print("\nBest parameters:")
print(rf_search.best_params_)

In [ ]:
best_rf_model = rf_search.best_estimator_

best_rf_cv_results = pd.DataFrame(
    rf_search.cv_results_
).sort_values("rank_test_score")

best_rf_cv_results[
    [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "params"
    ]
].head(10).round(4)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Tuned Random Forest"
    ],
    "Mean CV Macro F1": [
        cv_results["test_macro_f1"].mean(),
        rf_search.best_score_
    ],
    "CV Macro F1 Std": [
        cv_results["test_macro_f1"].std(),
        rf_search.cv_results_["std_test_score"][
            rf_search.best_index_
        ]
    ]
}).set_index("Model")

comparison.round(4)

In [ ]:

# Get the best model selected through cross-validation
final_model = rf_search.best_estimator_

# Make predictions on the untouched test set
y_test_pred = final_model.predict(X_test)

# Calculate evaluation metrics
final_accuracy = accuracy_score(y_test, y_test_pred)
final_balanced_accuracy = balanced_accuracy_score(y_test, y_test_pred)
final_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro"
)

print(f"Test Accuracy:          {final_accuracy:.4f}")
print(f"Test Balanced Accuracy: {final_balanced_accuracy:.4f}")
print(f"Test Macro F1-score:    {final_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        labels=["Dropout", "Enrolled", "Graduate"],
        zero_division=0
    )
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_test_pred,
    labels=["Dropout", "Enrolled", "Graduate"],
    display_labels=["Dropout", "Enrolled", "Graduate"],
    cmap="Blues",
    values_format="d"
)

plt.title("LearnWise AI — Final Model Evaluation")
plt.tight_layout()
plt.show()

In [ ]:
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "learnwise_model.joblib"

joblib.dump(final_model, MODEL_PATH)

print(f"Model saved successfully: {MODEL_PATH}")

In [ ]:
loaded_model = joblib.load(MODEL_PATH)

sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)

print("Actual outcome:    ", y_test.iloc[0])
print("Predicted outcome: ", prediction[0])

In [ ]:
import importlib
import app.predictor

importlib.reload(app.predictor)

from app.predictor import predict_student_outcome

In [ ]:
sample_student = X_test.iloc[0].to_dict()

result = predict_student_outcome(sample_student)

print("Prediction:", result["prediction"])

print("\nProbabilities:")

for outcome, probability in result["probabilities"].items():
    print(f"{outcome}: {probability}%")

In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

# Create feature configuration
feature_config = {
    "categorical": {},
    "numerical": {}
}

# Store categorical features and their available options
for column in categorical_columns:
    feature_config["categorical"][column] = sorted(
        [int(value) for value in X_train[column].dropna().unique()]
    )

# Store numerical features and their ranges
for column in numerical_columns:
    feature_config["numerical"][column] = {
        "min": float(X_train[column].min()),
        "max": float(X_train[column].max()),
        "default": float(X_train[column].median())
    }

# Save configuration
CONFIG_PATH = PROJECT_ROOT / "app" / "feature_config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(feature_config, file, indent=4)

print("Feature configuration saved!")
print("Categorical features:", len(feature_config["categorical"]))
print("Numerical features:", len(feature_config["numerical"]))
print("Total features:", len(feature_config["categorical"]) + len(feature_config["numerical"]))